# Parallel Execution Performance
Demonstrate speedup from parallel task execution

In [ ]:
import asyncio
import time
from typing import List, Dict
import pandas as pd

## Simulate Task Execution

In [ ]:
async def simulate_fetch_task(task_id: str, delay: float = 0.2):
    """Simulate fetching market data."""
    await asyncio.sleep(delay)
    return {"id": task_id, "result": f"Data for {task_id}", "time": delay}

async def simulate_price_task(task_id: str, delay: float = 0.1):
    """Simulate option pricing calculation."""
    await asyncio.sleep(delay)
    return {"id": task_id, "result": f"Price for {task_id}", "time": delay}

## Sequential Execution

In [ ]:
async def execute_sequential():
    """Execute tasks one by one."""
    tasks = [
        ('fetch_spot', simulate_fetch_task),
        ('fetch_vol', simulate_fetch_task),
        ('fetch_rate', simulate_fetch_task),
        ('price_call', simulate_price_task),
        ('price_put', simulate_price_task),
    ]
    
    start_time = time.time()
    results = []
    
    for task_id, task_func in tasks:
        result = await task_func(task_id)
        results.append(result)
        print(f"✓ {task_id} completed in {result['time']*1000:.0f}ms")
    
    total_time = time.time() - start_time
    return results, total_time

seq_results, seq_time = await execute_sequential()
print(f"\nSequential Total Time: {seq_time*1000:.0f}ms")

## Parallel Execution with Dependencies

In [ ]:
async def execute_parallel_with_deps():
    """Execute with proper dependency management."""
    start_time = time.time()
    
    # Group 1: Fetch data in parallel
    print("Group 1: Fetching data...")
    fetch_tasks = [
        simulate_fetch_task('fetch_spot'),
        simulate_fetch_task('fetch_vol'),
        simulate_fetch_task('fetch_rate')
    ]
    fetch_results = await asyncio.gather(*fetch_tasks)
    print(f"  ✓ All fetches complete in {max(r['time'] for r in fetch_results)*1000:.0f}ms")
    
    # Group 2: Price options in parallel (after data is fetched)
    print("\nGroup 2: Pricing options...")
    price_tasks = [
        simulate_price_task('price_call'),
        simulate_price_task('price_put')
    ]
    price_results = await asyncio.gather(*price_tasks)
    print(f"  ✓ All pricing complete in {max(r['time'] for r in price_results)*1000:.0f}ms")
    
    total_time = time.time() - start_time
    return fetch_results + price_results, total_time

par_results, par_time = await execute_parallel_with_deps()
print(f"\nParallel Total Time: {par_time*1000:.0f}ms")
print(f"Speedup: {seq_time/par_time:.2f}x")

## Iron Condor - 4 Leg Parallel Pricing

In [ ]:
async def price_iron_condor_sequential():
    """Price Iron Condor legs sequentially."""
    start = time.time()
    
    # Fetch data
    await simulate_fetch_task('fetch_spy', 0.2)
    
    # Price each leg one by one
    await simulate_price_task('put_440', 0.1)
    await simulate_price_task('put_445', 0.1)
    await simulate_price_task('call_455', 0.1)
    await simulate_price_task('call_460', 0.1)
    
    return time.time() - start

async def price_iron_condor_parallel():
    """Price Iron Condor legs in parallel."""
    start = time.time()
    
    # Fetch data
    await simulate_fetch_task('fetch_spy', 0.2)
    
    # Price all legs in parallel
    await asyncio.gather(
        simulate_price_task('put_440', 0.1),
        simulate_price_task('put_445', 0.1),
        simulate_price_task('call_455', 0.1),
        simulate_price_task('call_460', 0.1)
    )
    
    return time.time() - start

seq_ic_time = await price_iron_condor_sequential()
par_ic_time = await price_iron_condor_parallel()

print("Iron Condor Pricing:")
print("=" * 40)
print(f"Sequential: {seq_ic_time*1000:.0f}ms")
print(f"Parallel: {par_ic_time*1000:.0f}ms")
print(f"Speedup: {seq_ic_time/par_ic_time:.1f}x")
print(f"Time Saved: {(seq_ic_time - par_ic_time)*1000:.0f}ms")

## Scaling Analysis

In [ ]:
async def benchmark_scaling(n_legs: int):
    """Benchmark parallel vs sequential for N legs."""
    
    # Sequential
    seq_start = time.time()
    await simulate_fetch_task('fetch_data', 0.2)
    for i in range(n_legs):
        await simulate_price_task(f'leg_{i}', 0.1)
    seq_time = time.time() - seq_start
    
    # Parallel
    par_start = time.time()
    await simulate_fetch_task('fetch_data', 0.2)
    tasks = [simulate_price_task(f'leg_{i}', 0.1) for i in range(n_legs)]
    await asyncio.gather(*tasks)
    par_time = time.time() - par_start
    
    return seq_time, par_time

# Test with different numbers of legs
results = []
for n_legs in [2, 4, 8, 16]:
    seq, par = await benchmark_scaling(n_legs)
    results.append({
        'Legs': n_legs,
        'Sequential (ms)': f"{seq*1000:.0f}",
        'Parallel (ms)': f"{par*1000:.0f}",
        'Speedup': f"{seq/par:.1f}x"
    })

df = pd.DataFrame(results)
print("\nScaling Analysis:")
print("=" * 50)
print(df.to_string(index=False))

## Real-World Example: Complex Strategy

In [ ]:
async def execute_complex_strategy():
    """Execute a complex multi-asset strategy."""
    
    print("Executing Complex Strategy with Dependencies:")
    print("=" * 50)
    
    start = time.time()
    
    # Group 1: Fetch all market data
    print("\n[Group 1] Fetching market data...")
    data_tasks = [
        simulate_fetch_task('AAPL_data', 0.15),
        simulate_fetch_task('TSLA_data', 0.15),
        simulate_fetch_task('SPY_data', 0.15),
        simulate_fetch_task('volatility_surface', 0.25),
        simulate_fetch_task('risk_free_rate', 0.1)
    ]
    await asyncio.gather(*data_tasks)
    print(f"  ✓ Data fetched in {(time.time()-start)*1000:.0f}ms")
    
    # Group 2: Price all options
    group2_start = time.time()
    print("\n[Group 2] Pricing options...")
    price_tasks = [
        simulate_price_task('AAPL_call', 0.08),
        simulate_price_task('AAPL_put', 0.08),
        simulate_price_task('TSLA_call', 0.08),
        simulate_price_task('TSLA_put', 0.08),
        simulate_price_task('SPY_straddle', 0.12)
    ]
    await asyncio.gather(*price_tasks)
    print(f"  ✓ Options priced in {(time.time()-group2_start)*1000:.0f}ms")
    
    # Group 3: Calculate portfolio metrics
    group3_start = time.time()
    print("\n[Group 3] Calculating portfolio metrics...")
    await simulate_price_task('portfolio_greeks', 0.05)
    print(f"  ✓ Metrics calculated in {(time.time()-group3_start)*1000:.0f}ms")
    
    total = time.time() - start
    print(f"\n✅ Total execution time: {total*1000:.0f}ms")
    
    # Compare with sequential
    sequential_time = 0.15*3 + 0.25 + 0.1 + 0.08*4 + 0.12 + 0.05  # Sum all task times
    print(f"\nVs Sequential: {sequential_time*1000:.0f}ms")
    print(f"Speedup: {sequential_time/total:.1f}x")

await execute_complex_strategy()